[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/32_nlp/90_exercises.ipynb)

# Bài tập — Chương 32: NLP (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/32_nlp

In [ ]:
import math
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import NEWS_LEXICON, NEWS_SYNONYMS, make_news_vi
from ml_course.utils import repo_path

news = make_news_vi(4000, seed=0)
train, test = train_test_split(news, test_size=0.3, random_state=0, stratify=news.topic)


def strip_accents(s):
    s = unicodedata.normalize("NFD", s.lower()).replace("đ", "d")
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

## L0 — Nhớ khái niệm

**Câu 1.** (a) NFC và NFD khác nhau thế nào; vì sao phải chuẩn hoá? (b) Mô tả thuật toán BPE. (c) Công thức TF-IDF. (d) Giả thuyết phân bố. (e) BM25 khác TF-IDF ở hai điểm nào? (f) Định nghĩa Recall@k và MRR.

✍️ Trả lời:

**Đáp án:**
(a) NFC: ký tự có dấu là một code point dựng sẵn; NFD: chữ cái gốc + các dấu kết hợp. Hai chuỗi trông giống nhau nhưng khác byte → so sánh/đếm/khớp từ sai; chuẩn hoá NFC ngay khi đọc dữ liệu.
(b) Bắt đầu từ ký tự; lặp: đếm cặp ký hiệu kề nhau (theo tần suất từ), gộp cặp phổ biến nhất, ghi lại phép gộp; tokenize bằng cách áp dụng các phép gộp theo thứ tự.
(c) $\text{tf}(t, d)\cdot\text{idf}(t)$, $\text{idf} = \ln\frac{1+N}{1+\text{df}} + 1$ (scikit-learn), rồi chuẩn hoá L2.
(d) Từ xuất hiện trong ngữ cảnh giống nhau có nghĩa giống nhau → học vector từ từ văn bản không nhãn.
(e) Tần suất từ bão hoà (tham số $k_1$) và chuẩn hoá theo độ dài tài liệu (tham số $b$).
(f) Recall@k: tỉ lệ truy vấn có tài liệu đúng trong top-$k$; MRR: trung bình $1/$hạng của tài liệu đúng đầu tiên.

## L1 — Giải thích bằng lời

**Câu 2.** Một nhóm xây ô tìm kiếm sản phẩm cho sàn thương mại điện tử tiếng Việt: (1) bỏ stopword theo một danh sách có sẵn (gồm "không", "chưa", "rất"); (2) không chuẩn hoá Unicode vì "dữ liệu nhìn đều ổn"; (3) không xử lý truy vấn không dấu; (4) đánh giá bằng 10 truy vấn do chính họ nghĩ ra trong lúc dò lỗi, đạt Recall@1 = 100%. Chỉ ra các vấn đề và đề xuất.

✍️ Trả lời:

**Gợi ý:** (1) Bỏ "không", "chưa" làm mất phủ định ("không đường", "chưa qua sử dụng") — với tìm kiếm, TF-IDF/BM25 đã tự giảm trọng số từ phổ biến; nếu cần thì dùng danh sách stopword riêng cho miền. (2) Văn bản từ nhiều nguồn trộn NFC/NFD và hai kiểu bỏ dấu → cùng từ thành nhiều token, truy vấn không khớp (bài tập 7: accuracy giảm từ ~0.96 xuống ~0.6); chuẩn hoá NFC + thống nhất kiểu bỏ dấu ở cả chỉ mục và truy vấn. (3) Nhiều người dùng gõ không dấu → bỏ dấu (hoặc thêm trường không dấu) ở cả hai phía, n-gram ký tự cho lỗi chính tả. (4) 10 truy vấn tự nghĩ trong lúc dò lỗi = tối ưu trên chính tập đánh giá (bài tập 9 cho thấy cải thiện trên tập dev lớn hơn nhiều so với tập giữ lại); cần tập truy vấn thật từ log, đủ lớn, tách dev/test, đo Recall@k/MRR/nDCG, theo dõi cả truy vấn không có kết quả.

## L2 — Tính tay

**Câu 3.** Ba văn bản: $d_1$ = "giá vàng tăng", $d_2$ = "giá vàng giảm", $d_3$ = "điện thoại tăng giá". Theo công thức của scikit-learn (idf làm mịn, chuẩn hoá L2): tính `idf_vang` (idf của "vàng"), `idf_dien` (idf của "điện"), và `w_vang_d1` = trọng số TF-IDF của "vàng" trong $d_1$ sau chuẩn hoá.

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
idf_vang = math.log(4 / 3) + 1                                         # N = 3, df = 2
idf_dien = math.log(4 / 2) + 1                                         # df = 1
idf_gia, idf_tang = math.log(4 / 4) + 1, math.log(4 / 3) + 1           # "giá" có ở cả 3, "tăng" ở 2
w_vang_d1 = idf_vang / math.sqrt(idf_gia**2 + idf_vang**2 + idf_tang**2)
### END CODE HERE ###
print(f"idf(vàng) = {idf_vang:.4f} | idf(điện) = {idf_dien:.4f} | w(vàng, d1) = {w_vang_d1:.4f}")

In [ ]:
docs3 = ["giá vàng tăng", "giá vàng giảm", "điện thoại tăng giá"]
tv3 = TfidfVectorizer().fit(docs3)
check_close(idf_vang, tv3.idf_[tv3.vocabulary_["vàng"]], message="idf(vàng)")
check_close(idf_dien, tv3.idf_[tv3.vocabulary_["điện"]], message="idf(điện)")
check_close(w_vang_d1, tv3.transform(docs3[:1])[0, tv3.vocabulary_["vàng"]], message="trọng số sau chuẩn hoá L2")

**Câu 4.** (a) Ba truy vấn có tài liệu đúng ở hạng 1, 4 và 2. Tính `recall_at_1`, `recall_at_3`, `mrr`. (b) BM25 ($k_1 = 1.2$, $b = 0.75$) của truy vấn một từ $t$ với tài liệu $d$: $f(t, d) = 2$, $|d| = 10$, avgdl = 5, idf$(t)$ = 1.5. Tính `bm25`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
ranks4 = np.array([1, 4, 2])
recall_at_1, recall_at_3 = np.mean(ranks4 <= 1), np.mean(ranks4 <= 3)
mrr = np.mean(1 / ranks4)
bm25 = 1.5 * 2 * (1.2 + 1) / (2 + 1.2 * (1 - 0.75 + 0.75 * 10 / 5))
### END CODE HERE ###
print(f"R@1 = {recall_at_1:.3f}, R@3 = {recall_at_3:.3f}, MRR = {mrr:.4f} | BM25 = {bm25:.4f}")

In [ ]:
check_close([recall_at_1, recall_at_3, mrr], [1 / 3, 2 / 3, (1 + 1 / 4 + 1 / 2) / 3], message="Recall@k, MRR")
check_close(bm25, 6.6 / 4.1, message="BM25 (tài liệu dài gấp đôi trung bình bị phạt)")
check_less(bm25, 1.5 * (1.2 + 1), "BM25 bị chặn trên bởi idf·(k1 + 1) dù f lớn bao nhiêu")

## L3 — Code

**Bài 5.** Viết `normalize_vi(text, keep_accents=True)`: chuẩn hoá NFC, chữ thường, chuyển kiểu bỏ dấu **cũ** sang **mới** theo bảng `OLD_TO_NEW` (cho sẵn), gộp khoảng trắng; nếu `keep_accents=False` thì bỏ dấu (kể cả đ → d).

In [ ]:
OLD_TO_NEW = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa", "oè": "òe", "oé": "óe", "oẻ": "ỏe", "oẽ": "õe", "oẹ": "ọe",
              "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}


def normalize_vi(text, keep_accents=True):
    ### START CODE HERE ### (≈ 7 dòng)
    # STARTER: return text
    text = unicodedata.normalize("NFC", text).lower()
    for old, new in OLD_TO_NEW.items():
        text = text.replace(old, new)
    text = re.sub(r"\s+", " ", text).strip()
    if not keep_accents:
        text = strip_accents(text)
    return text
    ### END CODE HERE ###

In [ ]:
check_equal(normalize_vi(unicodedata.normalize("NFD", "Việt  Nam ")), "việt nam", "NFD → NFC, chữ thường, khoảng trắng")
check_equal(normalize_vi("Hoà bình, thuỷ lợi"), "hòa bình, thủy lợi", "kiểu bỏ dấu cũ → mới")
check_equal(normalize_vi("Hoà"), normalize_vi("Hòa"), "hai kiểu cho cùng kết quả")
check_equal(normalize_vi("Đường đi khó", keep_accents=False), "duong di kho", "bỏ dấu")
check_equal(normalize_vi(unicodedata.normalize("NFD", "Thuỷ"), keep_accents=False), "thuy", "NFD + bỏ dấu")

**Bài 6.** Cài **Multinomial Naive Bayes** cho văn bản: `fit(X, y, alpha)` với `X` ma trận đếm (thưa) → log prior của mỗi lớp và $\log P(\text{từ}\mid\text{lớp})$ với làm mịn Laplace: $\frac{N_{c,w} + \alpha}{N_c + \alpha|V|}$; `predict_log_proba(X)` trả log xác suất hậu nghiệm đã chuẩn hoá. So khớp `MultinomialNB`.

In [ ]:
class MyMultinomialNB:
    def fit(self, X, y, alpha=1.0):
        self.classes_ = np.unique(y)
        ### START CODE HERE ### (≈ 6 dòng)
        # STARTER: self.class_log_prior_ = None; self.feature_log_prob_ = None
        self.class_log_prior_ = np.log(np.array([np.mean(y == c) for c in self.classes_]))
        counts = np.vstack([np.asarray(X[y == c].sum(axis=0)).ravel() for c in self.classes_]) + alpha
        self.feature_log_prob_ = np.log(counts / counts.sum(axis=1, keepdims=True))
        ### END CODE HERE ###
        return self

    def predict_log_proba(self, X):
        ### START CODE HERE ### (≈ 3 dòng)
        # STARTER: return None
        joint = X @ self.feature_log_prob_.T + self.class_log_prior_
        return joint - np.logaddexp.reduce(joint, axis=1, keepdims=True)
        ### END CODE HERE ###

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

In [ ]:
cv6 = CountVectorizer().fit(train.text)
Xtr6, Xte6 = cv6.transform(train.text), cv6.transform(test.text)
mine6 = MyMultinomialNB().fit(Xtr6, train.topic.values, alpha=1.0)
ref6 = MultinomialNB(alpha=1.0).fit(Xtr6, train.topic.values)
check_close(mine6.predict_log_proba(Xte6[:50]), ref6.predict_log_proba(Xte6[:50]), tol=1e-8, message="khớp MultinomialNB")
acc6 = (mine6.predict(Xte6) == test.topic.values).mean()
print(f"accuracy = {acc6:.3f}")
check_greater(acc6, 0.9, "accuracy")

## L4 — Debug

**Bài 7.** Mô hình phân loại tin (cho sẵn) đạt ~0.96 trên tập test. Khi triển khai, văn bản đến từ một hệ thống khác (lưu dạng **NFD**) và accuracy tụt mạnh. Tái hiện lỗi, rồi sửa `make_vectorizer` để mô hình **bền** với dạng Unicode (gợi ý: tham số `preprocessor`).

In [ ]:
test_nfd = test.text.map(lambda s: unicodedata.normalize("NFD", s))


def make_vectorizer_buggy():
    return TfidfVectorizer(ngram_range=(1, 2))


def make_vectorizer():
    ### START CODE HERE ### (≈ 1 dòng)
    # STARTER: return TfidfVectorizer(ngram_range=(1, 2))
    return TfidfVectorizer(ngram_range=(1, 2), preprocessor=lambda s: unicodedata.normalize("NFC", s).lower())
    ### END CODE HERE ###


results7 = {}
for name, make in [("lỗi", make_vectorizer_buggy), ("đã sửa", make_vectorizer)]:
    vec = make().fit(train.text)
    clf = LinearSVC().fit(vec.transform(train.text), train.topic)
    results7[name] = ((clf.predict(vec.transform(test.text)) == test.topic).mean(), (clf.predict(vec.transform(test_nfd)) == test.topic).mean())
    print(f"vectorizer {name:6s}: test NFC = {results7[name][0]:.3f} | test NFD = {results7[name][1]:.3f}")
print("token hoá 'điện thoại' dạng NFD bởi bản lỗi:", make_vectorizer_buggy().build_analyzer()(unicodedata.normalize("NFD", "Điện thoại")))

In [ ]:
check_less(results7["lỗi"][1], results7["lỗi"][0] - 0.2, "tái hiện: NFD làm accuracy tụt mạnh")
check_close(results7["đã sửa"][1], results7["đã sửa"][0], tol=1e-9, message="đã sửa: NFD và NFC cho cùng kết quả")

**Nhận xét:** Ở dạng NFD, dấu là các code point riêng (loại Mn) mà regex `\w` của `token_pattern` **không** coi là chữ → "điện" bị cắt thành các mảnh như "đie", "n" — không khớp từ vựng đã học, mô hình như đọc văn bản lạ. Sửa bằng cách chuẩn hoá NFC trong `preprocessor` (chạy trước tokenizer, cho cả lúc fit và lúc dự đoán) — đặt bước chuẩn hoá **bên trong** pipeline để không thể quên khi triển khai. Bài học chung: dữ liệu production khác dữ liệu train theo những cách "vô hình" (encoding, định dạng) — kiểm tra phân phối đầu vào khi triển khai (chương 40).

## L5 — Phân tích

**Bài 8.** **Số chiều embedding.** Dựng ma trận PPMI (cửa sổ ±3) trên 20 000 tin không nhãn (code cho sẵn), rồi với $k \in \{2, 5, 10, 25, 50, 100, 200\}$ lấy embedding SVD $k$ chiều. Lưu `syn_top5[k]` (tỉ lệ cặp đồng nghĩa nằm trong top-5 láng giềng) và `acc30[k]` (accuracy phân loại chủ đề với 30 nhãn — trung bình embedding + LogReg, trung bình 5 lần chọn mẫu). Nhận xét.

In [ ]:
LEXICON = {tuple(w.lower().split()) for w in NEWS_LEXICON}
MAX_LEN = max(map(len, LEXICON))


def segment(text):
    syl = re.findall(r"\w[\w\-]*", unicodedata.normalize("NFC", text).lower())
    tokens, i = [], 0
    while i < len(syl):
        for L in range(min(MAX_LEN, len(syl) - i), 0, -1):
            if L == 1 or tuple(syl[i:i + L]) in LEXICON:
                tokens.append("_".join(syl[i:i + L]))
                i += L
                break
    return tokens


docs8 = [segment(t) for t in make_news_vi(20000, seed=1).text]
counts8 = Counter(w for d in docs8 for w in d)
vocab8 = sorted(w for w, c in counts8.items() if c >= 3)
w2i8 = {w: i for i, w in enumerate(vocab8)}
C8 = np.zeros((len(vocab8), len(vocab8)))
for d in docs8:
    ids = [w2i8[w] for w in d if w in w2i8]
    for i, c in enumerate(ids):
        for j in range(max(0, i - 3), min(len(ids), i + 4)):
            if j != i:
                C8[c, ids[j]] += 1
with np.errstate(divide="ignore"):
    ppmi8 = np.maximum(np.log(C8 * C8.sum() / C8.sum(1, keepdims=True) / C8.sum(0, keepdims=True)), 0)
U8, S8, _ = np.linalg.svd(ppmi8)
pairs8 = []
for group in NEWS_SYNONYMS:
    toks = [t for t in ("_".join(x.lower().split()) for x in group) if t in w2i8]
    pairs8 += [(w2i8[toks[i]], w2i8[toks[j]]) for i in range(len(toks)) for j in range(i + 1, len(toks))]
seg_train = [segment(t) for t in train.text]
seg_test = [segment(t) for t in test.text]

In [ ]:
syn_top5, acc30 = {}, {}
### START CODE HERE ### (≈ 16 dòng)
# STARTER: # với mỗi k: E = U8[:, :k] * sqrt(S8[:k]); chuẩn hoá hàng; đo top-5 cho pairs8; vector văn bản = trung bình; LogReg với 30 nhãn × 5 seed
for k in [2, 5, 10, 25, 50, 100, 200]:
    E = U8[:, :k] * np.sqrt(S8[:k])
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    sims = E @ E.T
    np.fill_diagonal(sims, -np.inf)
    top5 = np.argsort(-sims, axis=1)[:, :5]
    syn_top5[k] = np.mean([b in top5[a] or a in top5[b] for a, b in pairs8])

    def doc_vecs(seg_docs):
        return np.array([E[[w2i8[w] for w in d if w in w2i8]].mean(0) for d in seg_docs])

    Xtr8, Xte8 = doc_vecs(seg_train), doc_vecs(seg_test)
    scores = []
    for seed in range(5):
        idx = np.random.default_rng(seed).choice(len(Xtr8), 30, replace=False)
        lr = LogisticRegression(max_iter=3000, C=10).fit(Xtr8[idx], train.topic.values[idx])
        scores.append((lr.predict(Xte8) == test.topic.values).mean())
    acc30[k] = np.mean(scores)
    print(f"k = {k:3d}: đồng nghĩa trong top-5 = {syn_top5[k]:.2f} | accuracy 30 nhãn = {acc30[k]:.3f}")
### END CODE HERE ###

In [ ]:
check_greater(syn_top5[50], syn_top5[2] + 0.3, "k quá nhỏ không phân biệt được từ")
check_greater(acc30[50], acc30[2] + 0.05, "k quá nhỏ: phân loại kém")
check_greater(max(acc30.values()), 0.75, "có k cho accuracy 30 nhãn tốt")

**Nhận xét:** $k$ quá nhỏ (2–5 chiều) chỉ đủ phân tách vài hướng lớn (chủ đề) — từ trong cùng chủ đề dồn vào nhau, đồng nghĩa không nổi bật giữa các từ cùng chủ đề. Tăng $k$ cải thiện nhanh rồi bão hoà; $k$ rất lớn đưa cả các thành phần nhiễu (giá trị kỳ dị nhỏ) vào — với từ vựng chỉ vài trăm từ, lợi ích thêm gần như không có, và với 30 nhãn, nhiều chiều hơn còn làm LogReg dễ overfit. $k$ là siêu tham số: chọn bằng đánh giá nội tại (đồng nghĩa) **và** tác vụ đích; giá trị phổ biến cho word embedding thật là 100–300.

## L6 — Thực tế

**Bài 9.** **Mở rộng truy vấn cho FAQ.** Dùng FAQ của bài 04. Chia truy vấn: truy vấn **đầu tiên** của mỗi FAQ là tập **dev** (25 câu), còn lại là **test** (50 câu). (1) Baseline: TF-IDF + bỏ dấu → `base_dev`, `base_test` (Recall@1). (2) Xem các truy vấn dev bị sai, xây từ điển `SYNONYMS` (cụm người dùng hay nói → cụm trong FAQ, vd "ship" → "vận chuyển giao hàng"), viết `expand(query)` nối thêm các cụm thay thế khi truy vấn chứa khoá → `exp_dev`, `exp_test`. (3) So sánh mức cải thiện trên dev và test — giải thích.

In [ ]:
faq = pd.read_csv(repo_path("data", "sample", "faq_vi.csv"))
faq_q = pd.read_csv(repo_path("data", "sample", "faq_queries_vi.csv"))
faq_docs = [strip_accents(d) for d in faq.question + " " + faq.answer]
gold9 = (faq_q.faq_id - 1).values
is_dev = faq_q.groupby("faq_id").cumcount().values == 0
vec9 = TfidfVectorizer(sublinear_tf=True).fit(faq_docs)
D9 = vec9.transform(faq_docs)


def recall_at_1(queries, mask):
    S = (vec9.transform(queries) @ D9.T).toarray()
    return np.mean(S.argmax(1)[mask] == gold9[mask])


base_queries = [strip_accents(q) for q in faq_q["query"]]
base_dev, base_test = recall_at_1(base_queries, is_dev), recall_at_1(base_queries, ~is_dev)
print(f"baseline: R@1 dev = {base_dev:.3f} | test = {base_test:.3f}")
for q, g in zip(faq_q["query"][is_dev], gold9[is_dev]):
    S = (vec9.transform([strip_accents(q)]) @ D9.T).toarray()[0]
    if S.argmax() != g:
        print(f"  SAI: {q!r} → {faq.question[S.argmax()]!r} (đúng: {faq.question[g]!r})")

In [ ]:
### START CODE HERE ### (≈ 12 dòng)
# STARTER: SYNONYMS = {}  # "cụm người dùng nói": "cụm trong FAQ"
# STARTER: exp_dev, exp_test = None, None   # Recall@1 sau khi mở rộng truy vấn bằng SYNONYMS
SYNONYMS = {"trả lại": "đổi trả", "mấy ngày": "bao lâu nhận được", "tới": "nhận được", "ship": "vận chuyển giao hàng",
            "ở đâu": "tình trạng theo dõi", "bỏ đơn": "hủy đơn", "trả tiền": "thanh toán", "trả lại tiền": "hoàn tiền",
            "sửa": "bảo hành", "hỏng": "bảo hành hư hỏng", "vỡ": "hư hỏng lỗi", "nhầm": "sai", "chỗ": "địa chỉ",
            "không nhớ": "quên", "voucher": "mã giảm giá", "hóa đơn đỏ": "hóa đơn vat", "mỹ": "nước ngoài",
            "size": "kích cỡ", "hàng thật": "chính hãng", "trả sau": "trả góp", "tổng đài": "liên hệ chăm sóc khách hàng",
            "điểm thưởng": "tích điểm", "có hàng lại": "hết hàng có lại", "gấp": "hỏa tốc", "xem hàng": "kiểm tra"}
SYN_STRIPPED = {strip_accents(k): strip_accents(v) for k, v in SYNONYMS.items()}


def expand(query):
    q = strip_accents(query)
    extra = [v for k, v in SYN_STRIPPED.items() if re.search(r"\b" + re.escape(k) + r"\b", q)]
    return q + " " + " ".join(extra)


expanded = [expand(q) for q in faq_q["query"]]
exp_dev, exp_test = recall_at_1(expanded, is_dev), recall_at_1(expanded, ~is_dev)
### END CODE HERE ###
print(f"mở rộng: R@1 dev = {exp_dev:.3f} (+{exp_dev - base_dev:.3f}) | test = {exp_test:.3f} (+{exp_test - base_test:.3f})")

In [ ]:
check_greater(exp_dev, base_dev + 0.2, "cải thiện lớn trên dev (nơi xây từ điển)")
check_greater(exp_test, base_test - 1e-9, "không làm hại tập test")
check_less(exp_test - base_test, exp_dev - base_dev, "cải thiện trên test nhỏ hơn trên dev")

**Nhận xét:** Từ điển xây từ chính các lỗi của tập dev nên dev cải thiện rất mạnh — nhưng đó là **tối ưu trên tập đánh giá**, con số lạc quan. Trên 50 truy vấn test giữ lại, cải thiện nhỏ hơn hẳn (phần lớn truy vấn test đã đúng, và chỉ những cách nói có trong từ điển mới được lợi). Hai bài học: (1) luôn báo cáo trên tập **chưa dùng để thiết kế**; (2) từ điển đồng nghĩa thủ công rẻ và dễ kiểm soát nhưng không bao giờ phủ hết cách nói — thực tế kết hợp với sentence embedding (bài 04) và học từ log truy vấn–click. Lưu ý thêm: tập dev ở đây (câu diễn đạt lại, có dấu) khó hơn tập test (nhiều câu không dấu) — chia dev/test nên cùng phân phối.

## L7 — Nâng cao

**Bài 10.** **Topic modeling không giám sát với NMF.** Không dùng nhãn: biểu diễn 6 000 tin bằng TF-IDF (unigram + bigram, `min_df=3`, `max_df=0.3`), phân tích $X \approx WH$ bằng `NMF` (`init="nndsvda"`, `random_state=0`, `max_iter=800`) với $k$ = 6 và 12 chủ đề. Gán mỗi tin vào chủ đề có trọng số lớn nhất trong $W$; tính **purity** $\frac{1}{N}\sum_{\text{cụm}}\max_{\text{nhãn}}|\text{cụm}\cap\text{nhãn}|$ — với $k = 6$ so với nhãn `topic` (6 chủ đề lớn), với $k = 12$ so với `subtopic` (12 chủ đề con). Lưu `purity10[k]`; in 6 từ nặng nhất của mỗi chủ đề (hàng của $H$).

In [ ]:
big10 = make_news_vi(6000, seed=5)
purity10 = {}
### START CODE HERE ### (≈ 12 dòng)
# STARTER: # TfidfVectorizer(...) → X; với k ∈ {6, 12}: NMF → W, H; cụm = W.argmax(1); purity theo topic (k = 6) / subtopic (k = 12)
tv10 = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.3).fit(big10.text)
X10 = tv10.transform(big10.text)
feats10 = tv10.get_feature_names_out()
for k, label in [(6, "topic"), (12, "subtopic")]:
    nmf = NMF(k, init="nndsvda", random_state=0, max_iter=800)
    W = nmf.fit_transform(X10)
    cluster = W.argmax(1)
    table = pd.crosstab(cluster, big10[label])
    purity10[k] = table.max(axis=1).sum() / len(big10)
    print(f"k = {k:2d}: purity theo {label} = {purity10[k]:.3f} | số nhãn khác nhau là đa số của một cụm = {table.idxmax(axis=1).nunique()}")
    if k == 12:
        for t in range(k):
            print(f"   chủ đề {t:2d} ({table.loc[t].idxmax():12s}): {', '.join(feats10[np.argsort(-nmf.components_[t])[:6]])}")
### END CODE HERE ###

In [ ]:
check_greater(purity10[12], 0.8, "12 chủ đề NMF khớp tốt với 12 chủ đề con")
check_less(purity10[6], purity10[12] - 0.1, "6 chủ đề NMF khớp 6 chủ đề lớn kém hơn hẳn")

**Nhận xét:** NMF tìm ra các nhóm từ cùng xuất hiện — với $k = 12$ gần trùng với các chủ đề con thật dù không thấy nhãn nào. Với $k = 6$, NMF **không** tái tạo 6 chủ đề lớn: hai chủ đề con của cùng một chủ đề lớn (bóng đá / điền kinh) gần như không dùng chung từ nào, nên NMF tách một chủ đề lớn thành hai và gộp những thứ khác — cấu trúc "tự nhiên" theo đồng xuất hiện từ không nhất thiết trùng với cách con người gán nhãn. Để ý cũng có chủ đề bắt các **cụm khuôn mẫu** lặp lại ("nhiều hơn so với cùng kỳ") thay vì nội dung — cần lọc cụm khuôn mẫu/stopword miền trước khi diễn giải. Chọn $k$ như chọn số cụm (chương 20): xem độ mạch lạc của từ khoá (coherence), độ ổn định qua các seed, và mục đích sử dụng. Các mô hình chủ đề khác: LDA (xác suất), BERTopic (embedding + phân cụm).